In [ ]:
CATALOG = "fso_market_intelligence"
SCHEMA  = "frontier_labs"
DS      = "lmarena-ai/leaderboard-dataset"
SPLIT   = "latest"   # 'latest' = current board; 'full' = historical
SIGNALS = ["agent_task_outcome_explicit", "agent_steerability", "agent_tool_hallucination",
           "agent_praise_complaint", "agent_bash_recovery_steps"]

In [ ]:
import json, urllib.request, urllib.parse, time

def fetch_config(config, split=SPLIT):
    """Pull all rows of a dataset config/split via the HF datasets-server (keyless, paginated)."""
    base = "https://datasets-server.huggingface.co/rows"
    out, offset = [], 0
    while True:
        q = urllib.parse.urlencode({"dataset": DS, "config": config, "split": split, "offset": offset, "length": 100})
        req = urllib.request.Request(f"{base}?{q}", headers={"User-Agent": "Mozilla/5.0"})
        for a in range(4):
            try:
                d = json.loads(urllib.request.urlopen(req, timeout=60).read()); break
            except Exception as e:
                if a == 3: raise
                time.sleep(3 * (a + 1))
        rows = [r["row"] for r in d.get("rows", [])]
        out += rows
        total = d.get("num_rows_total", len(out))
        offset += len(rows)
        if not rows or offset >= total:
            break
    return out

agent = fetch_config("agent")
print("agent rows:", len(agent), "| publish_date:", agent[0].get("leaderboard_publish_date") if agent else None)

## 1. `arena_agent_leaderboard` — overall model ranking

In [ ]:
from pyspark.sql import functions as F

def to_df(rows):
    import pandas as pd
    return spark.createDataFrame(pd.DataFrame(rows))

lb = (to_df(agent)
      .withColumnRenamed("model_name", "model")
      .withColumnRenamed("score_ci_lower", "ci_lower")
      .withColumnRenamed("score_ci_upper", "ci_upper")
      .withColumnRenamed("leaderboard_publish_date", "publish_date")
      .withColumn("captured_at", F.current_date()))
(lb.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.arena_agent_leaderboard"))
print("arena_agent_leaderboard rows:", lb.count())
display(lb.orderBy("rank").limit(10))

## 2. `arena_agent_signals` — the 5 agent sub-dimensions

In [ ]:
sig_rows = []
for cfg in SIGNALS:
    signal = cfg.replace("agent_", "")
    for r in fetch_config(cfg):
        r = dict(r); r["signal"] = signal
        sig_rows.append(r)
    print(f"  {signal}: pulled")

sig = (to_df(sig_rows)
       .withColumnRenamed("model_name", "model")
       .withColumnRenamed("score_ci_lower", "ci_lower")
       .withColumnRenamed("score_ci_upper", "ci_upper")
       .withColumnRenamed("leaderboard_publish_date", "publish_date")
       .withColumn("captured_at", F.current_date()))
(sig.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.arena_agent_signals"))
print("arena_agent_signals rows:", sig.count())